# Modelo de Recomendação — Gêmeo Digital / OpenDC

**Topologia:** EC2 `m7i.xlarge` · **Versão:** 1.0.0 · **Paradigma:** otimização com restrições (não echo de simulação)

Arquitetura em 3 camadas:
1. **Posicionamento** (KNN euclidiano, k=3) — onde este estado está no espaço simulado?
2. **Predição** (regressão polinomial grau 1–2, validação LOO) — o que vai acontecer?
3. **Recomendação** (otimizador `min_cost_energy_under_sla_v1`) — o que fazer?

O modelo é serializado em `model_bundle/` como artefatos JSON puros, consumíveis por Node.js sem depedência Python em runtime.

> **Pré-requisito:** executar `scripts/collect_data.py` para gerar `datasets/opendc_consolidated.csv`.

In [ ]:
# ── Bloco 2: Imports ──────────────────────────────────────────────────────
import json
import statistics
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import LeaveOneOut

warnings.filterwarnings('ignore')
print('✅ Imports OK')

In [ ]:
# ── Bloco 3: Paths, versão e pricing ──────────────────────────────────────
PROJECT_ROOT  = Path('.')
DATASETS_DIR  = PROJECT_ROOT / 'datasets'
MODEL_DIR     = PROJECT_ROOT / 'model_bundle'
FIGURES_DIR   = PROJECT_ROOT / 'figures' / 'model'
PRICING_FILE  = PROJECT_ROOT / 'pricing' / 'pricing_metadata.json'

MODEL_VERSION = '1.0.0'
TOPOLOGY      = 'm7i.xlarge'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Preço EC2 — carregado de pricing_metadata.json
if PRICING_FILE.exists():
    _p = json.loads(PRICING_FILE.read_text(encoding='utf-8'))
    EC2_PRICE_USD_PER_H = float(_p.get('price_usd_per_h', 0.1904))
    PRICING_CITATION    = _p.get('citation', 'AWS EC2 Pricing')
    PRICING_BIBTEX      = _p.get('bibtex', '')
else:
    EC2_PRICE_USD_PER_H = 0.1904
    PRICING_CITATION    = 'AWS EC2 Pricing'
    PRICING_BIBTEX      = ''

print(f'PROJECT_ROOT : {PROJECT_ROOT.resolve()}')
print(f'EC2_PRICE    : ${EC2_PRICE_USD_PER_H}/h')
print(f'MODEL_VERSION: {MODEL_VERSION}')

In [ ]:
# ── Bloco 4: Carregar e validar dataset ───────────────────────────────────
CSV_PATH = DATASETS_DIR / 'opendc_consolidated.csv'
assert CSV_PATH.exists(), (
    f'Dataset não encontrado: {CSV_PATH}\n'
    'Execute: python scripts/collect_data.py'
)

df = pd.read_csv(CSV_PATH)
print(f'Dataset carregado: {len(df)} cenários × {len(df.columns)} colunas')

assert len(df) == 7,  f'Esperados 7 cenários, encontrados {len(df)}'
assert df['n_unique_tasks'].is_monotonic_increasing, 'Cenários fora de ordem crescente de tasks'
assert df['completion_rate_pct'].min() >= 99.0, 'completion_rate < 99% em algum cenário'

print('\n✅ Validações OK')
print(df[['scenario', 'n_unique_tasks', 'energy_total_kwh', 'exec_mean_s', 'wait_mean_s',
          'cpu_util_mean_pct', 'completion_rate_pct']].to_string(index=False))

In [ ]:
# ── Bloco 5: Feature engineering e StandardScaler ─────────────────────────
FEATURE_COLS = ['n_unique_tasks', 'exec_mean_s', 'sim_duration_h']
TARGET_COLS  = [
    'energy_total_kwh', 'wait_mean_s', 'estimated_cost_usd',
    'cpu_util_mean_pct', 'saturation_pct', 'completion_rate_pct'
]

X = df[FEATURE_COLS].values.astype(float)
y = df[TARGET_COLS].values.astype(float)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaler_json = {
    'feature_cols': FEATURE_COLS,
    'mean_':        scaler.mean_.tolist(),
    'scale_':       scaler.scale_.tolist(),
    'var_':         scaler.var_.tolist(),
    'n_samples':    int(scaler.n_samples_seen_),
}

print('StandardScaler fit:')
for i, col in enumerate(FEATURE_COLS):
    print(f'  {col:<22}: mean={scaler.mean_[i]:.4f}, scale={scaler.scale_[i]:.4f}')

In [ ]:
# ── Bloco 6: Preditor — regressão polinomial + LOO ────────────────────────
x_primary = df['n_unique_tasks'].values.astype(float)

def loo_mae(x, y, degree):
    loo = LeaveOneOut()
    errors = []
    for tr, te in loo.split(x):
        coeffs = np.polyfit(x[tr], y[tr], degree)
        pred   = np.polyval(coeffs, x[te])
        errors.append(abs(pred[0] - y[te][0]))
    return float(np.mean(errors))

predictor_json = {}
df_val_rows    = []

for target in TARGET_COLS:
    y_t   = df[target].values.astype(float)
    maes  = {d: loo_mae(x_primary, y_t, d) for d in [1, 2]}
    best  = min(maes, key=maes.get)
    coeffs = np.polyfit(x_primary, y_t, best)
    y_hat  = np.polyval(coeffs, x_primary)
    r2_val = float(r2_score(y_t, y_hat))

    predictor_json[target] = {
        'coefficients':  coeffs.tolist(),
        'degree':        best,
        'input_feature': 'n_unique_tasks',
        'mae_loo':       maes[best],
        'r2_train':      r2_val,
        'y_range':       [float(y_t.min()), float(y_t.max())],
        'x_range':       [float(x_primary.min()), float(x_primary.max())],
    }
    df_val_rows.append({
        'target': target, 'degree': best,
        'mae_loo': round(maes[best], 6), 'r2_train': round(r2_val, 4)
    })
    print(f'  {target:<26}: grau={best}, MAE_LOO={maes[best]:.6f}, R2={r2_val:.4f}')

df_val = pd.DataFrame(df_val_rows).set_index('target')
print('\n✅ Preditor OK')

In [ ]:
# ── Bloco 7: Posicionador KNN (numpy puro, euclidiano, k=3) ───────────────
N_NEIGHBORS = min(3, len(df))

positioner_json = {
    'n_neighbors':  N_NEIGHBORS,
    'metric':       'euclidean',
    'feature_cols': FEATURE_COLS,
    'scenarios':    df['scenario'].tolist(),
    'X_scaled':     X_scaled.tolist(),
    'X_raw':        X.tolist(),
}

print(f'KNN positioner: k={N_NEIGHBORS}, {len(df)} cenários de referência')
print(f'Scenarios: {df["scenario"].tolist()}')

In [ ]:
# ── Bloco 8: scenario_db + rules_json ─────────────────────────────────────
scenario_db = []
for _, row in df.iterrows():
    scenario_db.append({
        'scenario':           row['scenario'],
        'n_tasks':            int(row['n_unique_tasks']),
        'sim_duration_h':     float(row['sim_duration_h']),
        'n_hosts':            int(row['n_hosts']),
        'energy_total_kwh':   float(row['energy_total_kwh']),
        'power_mean_w':       float(row.get('power_mean_w', 70.0)),
        'cpu_util_mean_pct':  float(row['cpu_util_mean_pct']),
        'wait_mean_s':        float(row['wait_mean_s']),
        'wait_p95_s':         float(row.get('wait_p95_s', 0.0)),
        'exec_mean_s':        float(row['exec_mean_s']),
        'estimated_cost_usd': float(row['estimated_cost_usd']),
        'saturation_pct':     float(row['saturation_pct']),
        'completion_rate_pct':float(row['completion_rate_pct']),
        'tasks_per_kwh':      float(row.get('tasks_per_kwh', 0.0)),
        'cost_per_task_usd':  float(row.get('cost_per_task_usd', 0.0)),
    })

rules_json = {
    'description': 'Thresholds derivados dos 7 cenarios simulados na topologia m7i.xlarge',
    'topology': TOPOLOGY,
    'status_thresholds': {
        'cpu_util_pct': {
            'ok':        float(df['cpu_util_mean_pct'].quantile(0.33)),
            'attention': float(df['cpu_util_mean_pct'].quantile(0.66)),
            'critical':  float(df['cpu_util_mean_pct'].quantile(0.90)),
            'unit': 'percent'
        },
        'wait_mean_s': {
            'ok':        float(df['wait_mean_s'].quantile(0.33)),
            'attention': float(df['wait_mean_s'].quantile(0.66)),
            'critical':  float(df['wait_mean_s'].quantile(0.90)),
            'unit': 'seconds'
        },
        'saturation_pct': {
            'ok': 0.0, 'attention': 5.0, 'critical': 15.0, 'unit': 'percent'
        },
        'completion_rate_pct': {
            'ok': 99.0, 'attention': 95.0, 'critical': 90.0,
            'unit': 'percent', 'direction': 'lower_is_worse'
        },
    },
    'capacity': {
        'max_tasks_observed':      int(df['n_unique_tasks'].max()),
        'max_energy_kwh_observed': float(df['energy_total_kwh'].max()),
        'max_cost_usd_observed':   float(df['estimated_cost_usd'].max()),
        'n_hosts':                 int(df['n_hosts'].max()),
    },
    'action_templates': {
        'scale_up':       'Scale up: increase active hosts -- load approaching observed maximum capacity',
        'scale_down':     'Scale down: reduce active hosts -- cluster oversized for current load',
        'energy_ok':      'Energy consumption within safe envelope',
        'energy_warning': 'Energy consumption above 66th percentile of simulated scenarios',
        'latency_ok':     'Wait time within normal parameters',
        'latency_high':   'Wait time elevated -- review scheduling policy',
        'cost_ok':        'Estimated cost within configured SLA budget',
        'cost_over':      'Estimated cost exceeds configured SLA budget',
        'completion_ok':  'Completion rate nominal -- no action required',
        'completion_low': 'Completion rate below expected -- investigate task failures',
    },
}

print(f'scenario_db: {len(scenario_db)} cenarios')
print(f'max_tasks_observed: {rules_json["capacity"]["max_tasks_observed"]}')

In [ ]:
# ── Bloco 9 (Célula 18 original): optimizer.json ──────────────────────────
# Lê constantes da topologia e dos workloads gerados.

_topo_path = PROJECT_ROOT / 'topologies' / '1.first_experiment' / 'm7ixlarge.json'
assert _topo_path.exists(), f'Topologia não encontrada: {_topo_path}'

_topo = json.loads(_topo_path.read_text(encoding='utf-8'))
_host = _topo['clusters'][0]['hosts'][0]

CORES_PER_HOST   = int(_host['cpu']['coreCount'])       # 4
N_HOSTS_TOPOLOGY = int(_host['count'])                  # 100
IDLE_POWER_W     = float(_host['cpuPowerModel']['idlePower'])  # 70.0
MAX_POWER_W      = float(_host['cpuPowerModel']['maxPower'])   # 200.0

# avg_cores_per_task — média dos summary.json gerados pelos workloads
_summaries = sorted((PROJECT_ROOT / 'workload_traces' / 'generated_workloads').glob('*/summary.json'))
_cores = [float(json.loads(open(f, encoding='utf-8').read())['avg_cpu_count']) for f in _summaries]
AVG_CORES_PER_TASK = round(statistics.mean(_cores), 4) if _cores else float(CORES_PER_HOST)

# Checagem: idle por host derivado dos dados simulados (≈ 70 W)
_derived = [
    s['energy_total_kwh'] / s['sim_duration_h'] / s['n_hosts']
    for s in scenario_db if s['sim_duration_h'] > 0
]
DERIVED_IDLE_KW = round(statistics.median(_derived), 6) if _derived else IDLE_POWER_W / 1000

TARGET_UTIL = 0.70

optimizer_json = {
    'method': 'min_cost_energy_under_sla_v1',
    'description': (
        'Dado um workload, calcula o menor n. de hosts que atende a '
        'demanda de compute na banda-alvo de utilizacao, minimizando '
        'custo+energia sob SLA. Custo = hosts * duracao_h * price; '
        'energia = power(util) * hosts * duracao_h.'
    ),
    'topology':              TOPOLOGY,
    'cores_per_host':        CORES_PER_HOST,
    'n_hosts_topology':      N_HOSTS_TOPOLOGY,
    'max_hosts':             N_HOSTS_TOPOLOGY,
    'min_hosts':             1,
    'idle_power_w':          IDLE_POWER_W,
    'max_power_w':           MAX_POWER_W,
    'power_per_host_idle_kw': round(IDLE_POWER_W / 1000, 6),
    'power_per_host_idle_kw_derived': DERIVED_IDLE_KW,
    'price_usd_per_h':       EC2_PRICE_USD_PER_H,
    'avg_cores_per_task':    AVG_CORES_PER_TASK,
    'target_util':           TARGET_UTIL,
    'sla_defaults': {
        'min_completion_rate_pct': 99.0,
        'max_wait_s':              0.55,
        'max_saturation_pct':      5.0,
    },
    'demand_model': 'littles_law: concurrency = n_tasks * exec_mean_s / window_s',
    'notes': (
        'Os 7 cenarios simulados rodaram com 100 hosts fixos e utilizacao '
        '~0% (evidencia de sobre-provisionamento). O otimizador dimensiona '
        'o cluster a partir da demanda de compute estimada, nao por '
        'regressao sobre n. de hosts.'
    ),
}

(MODEL_DIR / 'optimizer.json').write_text(
    json.dumps(optimizer_json, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('optimizer.json gerado:')
print(f'  cores_per_host:      {CORES_PER_HOST}')
print(f'  n_hosts_topology:    {N_HOSTS_TOPOLOGY}')
print(f'  idle_power_w:        {IDLE_POWER_W}')
print(f'  max_power_w:         {MAX_POWER_W}')
print(f'  avg_cores_per_task:  {AVG_CORES_PER_TASK}')
print(f'  price_usd_per_h:     {EC2_PRICE_USD_PER_H}')
print(f'  target_util:         {TARGET_UTIL}')
print(f'  derived_idle_kw:     {DERIVED_IDLE_KW}')

In [ ]:
# ── Bloco 10 (Célula 20 original): Classe RecommendationModel ─────────────

class RecommendationModel:
    """
    Motor de recomendacao serializavel para datacenter OpenDC.
        model = RecommendationModel.load('model_bundle/')
        result = model.recommend(state={...}, sla={...})
    """

    def __init__(self, scaler_params, predictor_params, positioner_params,
                 scenario_db, rules, metadata, optimizer=None):
        self.scaler_params     = scaler_params
        self.predictor_params  = predictor_params
        self.positioner_params = positioner_params
        self.scenario_db       = {s['scenario']: s for s in scenario_db}
        self.rules             = rules
        self.metadata          = metadata
        self.optimizer         = optimizer or {}
        self._feature_cols     = scaler_params['feature_cols']
        self._mean             = np.array(scaler_params['mean_'])
        self._scale            = np.array(scaler_params['scale_'])
        self._X_scaled         = np.array(positioner_params['X_scaled'])
        self._scenarios_list   = positioner_params['scenarios']

    # ── Serialização ─────────────────────────────────────────────────────────
    def save(self, bundle_dir):
        d = Path(bundle_dir)
        d.mkdir(parents=True, exist_ok=True)
        (d / 'scaler.json').write_text(
            json.dumps(self.scaler_params, indent=2, ensure_ascii=False), encoding='utf-8')
        (d / 'predictor.json').write_text(
            json.dumps(self.predictor_params, indent=2, ensure_ascii=False), encoding='utf-8')
        (d / 'positioner.json').write_text(
            json.dumps(self.positioner_params, indent=2, ensure_ascii=False), encoding='utf-8')
        (d / 'scenario_db.json').write_text(
            json.dumps(list(self.scenario_db.values()), indent=2, ensure_ascii=False), encoding='utf-8')
        (d / 'rules.json').write_text(
            json.dumps(self.rules, indent=2, ensure_ascii=False), encoding='utf-8')
        (d / 'model_metadata.json').write_text(
            json.dumps(self.metadata, indent=2, ensure_ascii=False), encoding='utf-8')
        if self.optimizer:
            (d / 'optimizer.json').write_text(
                json.dumps(self.optimizer, indent=2, ensure_ascii=False), encoding='utf-8')
        # example_recommendation.json
        example_state = {
            'n_tasks_active': 110, 'exec_mean_s': 0.22, 'sim_duration_h': 0.0015,
            'energy_consumed_kwh': 0.011, 'wait_mean_s': 0.44, 'hosts_active': 100
        }
        example_sla = {'max_energy_kwh': 0.015, 'max_wait_s': 0.55, 'max_cost_usd': 0.008}
        try:
            (d / 'example_recommendation.json').write_text(
                json.dumps(self.recommend(example_state, example_sla), indent=2, ensure_ascii=False),
                encoding='utf-8')
        except Exception as exc:
            print(f'   ⚠  example_recommendation.json nao gerado: {exc}')
        print(f'✅ Modelo salvo em: {d.resolve()}')
        for f in sorted(d.glob('*.json')):
            print(f'   ├── {f.name:<35s} ({f.stat().st_size/1024:.1f} KB)')

    @classmethod
    def load(cls, bundle_dir):
        d = Path(bundle_dir)
        optimizer = None
        opt_path  = d / 'optimizer.json'
        if opt_path.exists():
            optimizer = json.loads(opt_path.read_text(encoding='utf-8'))
        return cls(
            scaler_params     = json.loads((d / 'scaler.json').read_text(encoding='utf-8')),
            predictor_params  = json.loads((d / 'predictor.json').read_text(encoding='utf-8')),
            positioner_params = json.loads((d / 'positioner.json').read_text(encoding='utf-8')),
            scenario_db       = json.loads((d / 'scenario_db.json').read_text(encoding='utf-8')),
            rules             = json.loads((d / 'rules.json').read_text(encoding='utf-8')),
            metadata          = json.loads((d / 'model_metadata.json').read_text(encoding='utf-8')),
            optimizer         = optimizer,
        )

    def _scale_input(self, x_raw):
        return (x_raw - self._mean) / self._scale

    # ── Camada 1: Posicionamento (KNN) ────────────────────────────────────────
    def _position(self, x_scaled):
        dists   = np.linalg.norm(self._X_scaled - x_scaled, axis=1)
        order   = np.argsort(dists)
        nearest = self._scenarios_list[order[0]]
        neighbor= self._scenarios_list[order[1]] if len(order) > 1 else None
        third   = self._scenarios_list[order[2]] if len(order) > 2 else None
        max_d   = dists.max() if dists.max() > 0 else 1
        sim_pct = float((1 - dists[order[0]] / max_d) * 100)
        return {
            'nearest_scenario': nearest,
            'similarity_pct':   round(sim_pct, 2),
            'neighbors':        [s for s in [neighbor, third] if s],
            'distances':        {self._scenarios_list[i]: round(float(dists[i]), 4) for i in order},
        }

    # ── Camada 2: Predição ────────────────────────────────────────────────────
    def _predict(self, n_tasks, state):
        preds = {}
        for target, info in self.predictor_params.items():
            val = np.polyval(info['coefficients'], n_tasks)
            val = float(np.clip(val, info['y_range'][0] * 0.5, info['y_range'][1] * 2.0))
            preds[target] = round(val, 8)
        # Custo recalculado por hosts reais (mais preciso que regressão sobre 100 hosts)
        n_hosts_active = state.get('hosts_active', None)
        sim_duration_h = state.get('sim_duration_h', 0)
        price_per_h    = self.metadata.get('pricing', {}).get('price_usd_per_h', 0.1904)
        if n_hosts_active and sim_duration_h > 0:
            preds['estimated_cost_usd'] = round(
                n_hosts_active * sim_duration_h * price_per_h, 8)
        return preds

    # ── Camada 3: Otimizador (NOVO) ────────────────────────────────────────────
    def find_optimal_config(self, state, sla=None):
        """Menor nº de hosts que atende a demanda na banda-alvo, sob SLA.
        Método: min_cost_energy_under_sla_v1 (Lei de Little + modelo linear de potência)."""
        opt = self.optimizer
        if not opt:
            return {'error': 'optimizer.json nao carregado'}
        sla = sla or {}

        cores_per_host     = float(opt['cores_per_host'])
        idle_power_w       = float(opt['idle_power_w'])
        max_power_w        = float(opt['max_power_w'])
        price_usd_per_h    = float(opt['price_usd_per_h'])
        avg_cores_per_task = float(opt['avg_cores_per_task'])
        target_util        = float(opt['target_util'])
        min_hosts          = int(opt['min_hosts'])
        max_hosts          = int(opt['max_hosts'])

        def power_kw(util):
            return (idle_power_w + (max_power_w - idle_power_w) * util) / 1000.0

        duration_h  = float(state.get('sim_duration_h', 0))
        window_s    = duration_h * 3600.0
        n_tasks     = float(state.get('n_tasks_active', 0))
        exec_mean   = float(state.get('exec_mean_s', 0))
        concurrency = (n_tasks * exec_mean) / window_s if window_s > 0 else n_tasks
        cores_required = concurrency * avg_cores_per_task

        sized_hosts = int(np.ceil(cores_required / (cores_per_host * target_util)))
        hard_floor  = int(np.ceil(cores_required / cores_per_host))
        recommended = max(sized_hosts, hard_floor, min_hosts)
        recommended = min(recommended, max_hosts)

        current_hosts   = int(state.get('hosts_active', 0)) or max_hosts
        util_at_rec     = (cores_required / (recommended * cores_per_host)) * 100.0 if recommended > 0 else 0.0
        util_current    = min(cores_required / (current_hosts * cores_per_host), 1.0) if current_hosts > 0 else 0.0

        cost_current  = current_hosts * duration_h * price_usd_per_h
        cost_opt      = recommended   * duration_h * price_usd_per_h
        energy_current = power_kw(util_current) * current_hosts * duration_h
        energy_opt     = power_kw(target_util)  * recommended   * duration_h

        feasible        = (util_at_rec <= 100.0) and (min_hosts <= recommended <= max_hosts)
        savings_hosts   = current_hosts - recommended
        savings_pct     = round((savings_hosts / current_hosts) * 100, 2) if current_hosts > 0 else 0.0

        return {
            'method':             opt.get('method', 'min_cost_energy_under_sla_v1'),
            'current_config':     {'hosts': current_hosts},
            'recommended_config': {'hosts': recommended},
            'demand': {
                'concurrency_tasks':              round(concurrency, 4),
                'cores_required':                 round(cores_required, 4),
                'utilization_at_recommended_pct': round(util_at_rec, 2),
            },
            'projected': {
                'cost_usd':             round(cost_opt, 8),
                'energy_kwh':           round(energy_opt, 8),
                'wait_mean_s':          state.get('wait_mean_s', None),
                'completion_rate_pct':  100.0 if feasible else 0.0,
            },
            'savings': {
                'cost_usd':   round(cost_current - cost_opt, 8),
                'energy_kwh': round(energy_current - energy_opt, 8),
                'hosts':      savings_hosts,
                'pct':        savings_pct,
            },
            'feasible': feasible,
        }

    # ── Camada 3: Recomendações (orientadas pelo otimizador) ──────────────────
    def _make_recommendations(self, state, predictions, positioning, sla, optimization=None):
        recs, warns, status = [], [], 'ok'
        thresholds = self.rules['status_thresholds']
        templates  = self.rules['action_templates']
        capacity   = self.rules['capacity']
        priority   = 1

        def escalate(s):
            nonlocal status
            if s == 'critical' or status == 'critical': status = 'critical'
            elif s == 'attention': status = 'attention'

        # Capacity — orientado pelo gap atual → ótimo
        if optimization and 'recommended_config' in optimization:
            cur  = optimization['current_config']['hosts']
            rec  = optimization['recommended_config']['hosts']
            sh   = optimization['savings']['hosts']
            sp   = optimization['savings']['pct']
            sc   = optimization['savings']['cost_usd']
            se   = optimization['savings']['energy_kwh']
            feas = optimization['feasible']
            up   = optimization['demand']['utilization_at_recommended_pct']
            if sh > 0:
                escalate('attention')
                recs.append({'priority': priority, 'category': 'capacity',
                    'action': templates.get('scale_down'),
                    'detail': f'Current: {cur} hosts | Optimal: {rec} hosts | Util at optimal: {up:.1f}%',
                    'impact': f'Savings: {sh} hosts ({sp:.1f}%) | Cost: -${sc:.5f} | Energy: -{se:.5f} kWh'})
            elif sh < 0:
                escalate('critical')
                warns.append(f'Cluster under-provisioned: need {rec} hosts, currently {cur}')
                recs.append({'priority': priority, 'category': 'capacity',
                    'action': templates.get('scale_up'),
                    'detail': f'Current: {cur} hosts | Optimal: {rec} hosts | Util at optimal: {up:.1f}%',
                    'impact': f'Missing {abs(sh)} hosts -- SLA at risk (feasible={feas})'})
            else:
                escalate('ok')
                recs.append({'priority': priority, 'category': 'capacity',
                    'action': 'Cluster at optimal configuration -- no scaling action required',
                    'detail': f'{cur} hosts | Util at optimal: {up:.1f}%',
                    'impact': 'Cost and energy minimized under SLA'})
        else:
            # Fallback (sem otimizador)
            n_in  = state.get('n_tasks_active', 0)
            h_act = state.get('hosts_active', capacity['n_hosts'])
            hdroom = capacity['max_tasks_observed'] - n_in
            if n_in < capacity['max_tasks_observed'] * 0.3:
                escalate('ok')
                recs.append({'priority': priority, 'category': 'capacity',
                    'action': templates['scale_down'],
                    'detail': f'Only {n_in} active tasks -- {h_act} hosts can be reduced',
                    'impact': f'Headroom: {hdroom} tasks until observed maximum'})
            elif n_in > capacity['max_tasks_observed'] * 0.85:
                escalate('critical')
                warns.append('Load approaching observed maximum capacity')
                recs.append({'priority': priority, 'category': 'capacity',
                    'action': templates['scale_up'],
                    'detail': f'{n_in} active tasks -- near limit of {capacity["max_tasks_observed"]}',
                    'impact': 'Risk of saturation and SLA degradation'})
            else:
                escalate('ok')
                recs.append({'priority': priority, 'category': 'capacity',
                    'action': 'Load within normal operational envelope',
                    'detail': f'{n_in} active tasks', 'impact': f'Headroom: {hdroom} tasks'})
        priority += 1

        # Energia
        pe  = predictions.get('energy_total_kwh', 0)
        se_ = sla.get('max_energy_kwh', float('inf'))
        if pe > se_:
            escalate('critical')
            warns.append(f'Predicted energy ({pe:.5f} kWh) exceeds SLA limit ({se_:.5f} kWh)')
            recs.append({'priority': priority, 'category': 'energy',
                'action': templates['energy_warning'],
                'detail': f'Predicted: {pe:.5f} kWh | SLA limit: {se_:.5f} kWh',
                'impact': f'Excess: {(pe-se_):.5f} kWh'})
        else:
            escalate('ok')
            margin = (1 - pe/se_)*100 if se_ < float('inf') else None
            recs.append({'priority': priority, 'category': 'energy',
                'action': templates['energy_ok'],
                'detail': f'Predicted: {pe:.5f} kWh',
                'impact': f'Energy margin: {margin:.1f}%' if margin else 'No SLA energy limit configured'})
        priority += 1

        # Latência
        pw  = predictions.get('wait_mean_s', 0)
        sw  = sla.get('max_wait_s', float('inf'))
        tw  = thresholds['wait_mean_s']
        if pw > sw or pw > tw['critical']:
            escalate('critical')
            warns.append(f'Predicted wait ({pw:.3f}s) exceeds critical threshold')
            recs.append({'priority': priority, 'category': 'latency',
                'action': templates['latency_high'],
                'detail': f'Predicted: {pw:.3f}s | SLA: {sw:.3f}s | Critical: {tw["critical"]:.3f}s',
                'impact': 'Experience degradation -- review scheduling policy'})
        elif pw > tw['attention']:
            escalate('attention')
            recs.append({'priority': priority, 'category': 'latency',
                'action': templates['latency_high'],
                'detail': f'Predicted: {pw:.3f}s -- above 66th percentile ({tw["attention"]:.3f}s)',
                'impact': 'Monitor -- may escalate to critical if load increases'})
        else:
            escalate('ok')
            recs.append({'priority': priority, 'category': 'latency',
                'action': templates['latency_ok'],
                'detail': f'Predicted: {pw:.3f}s -- within normal range', 'impact': 'No action required'})
        priority += 1

        # Custo
        pc  = predictions.get('estimated_cost_usd', 0)
        sco = sla.get('max_cost_usd', float('inf'))
        if pc > sco:
            escalate('attention')
            warns.append(f'Predicted cost (${pc:.4f}) exceeds SLA budget (${sco:.4f})')
            recs.append({'priority': priority, 'category': 'cost',
                'action': templates['cost_over'],
                'detail': f'Predicted: ${pc:.4f} | Limit: ${sco:.4f}',
                'impact': f'Excess: ${(pc-sco):.4f}'})
        else:
            recs.append({'priority': priority, 'category': 'cost',
                'action': templates['cost_ok'],
                'detail': f'Predicted: ${pc:.4f}', 'impact': 'Within budget'})
        priority += 1

        # Confiabilidade
        pcomp = predictions.get('completion_rate_pct', 100)
        tc    = thresholds['completion_rate_pct']
        if pcomp < tc['critical']:
            escalate('critical')
            warns.append(f'Predicted completion ({pcomp:.1f}%) below critical ({tc["critical"]}%)')
            recs.append({'priority': priority, 'category': 'reliability',
                'action': templates['completion_low'],
                'detail': f'Predicted: {pcomp:.1f}%', 'impact': 'Investigate task failures'})
        else:
            recs.append({'priority': priority, 'category': 'reliability',
                'action': templates['completion_ok'],
                'detail': f'Predicted: {pcomp:.1f}%', 'impact': 'No action required'})

        return recs, warns, status

    # ── Interface principal ────────────────────────────────────────────────────
    def recommend(self, state, sla=None):
        if sla is None: sla = {}
        x_raw = np.array([
            state.get('n_tasks_active', 0),
            state.get('exec_mean_s', 0),
            state.get('sim_duration_h', 0),
        ], dtype=float)
        x_scaled     = self._scale_input(x_raw)
        positioning  = self._position(x_scaled)
        predictions  = self._predict(x_raw[0], state)
        optimization = self.find_optimal_config(state, sla)
        recs, warns, status = self._make_recommendations(
            state, predictions, positioning, sla, optimization)
        cap = self.rules['capacity']
        n   = state.get('n_tasks_active', 0)
        headroom = {
            'tasks_until_max_observed': max(0, cap['max_tasks_observed'] - n),
            'energy_margin_pct': round(
                max(0, (1 - predictions['energy_total_kwh'] / cap['max_energy_kwh_observed']) * 100), 1
            ) if cap['max_energy_kwh_observed'] > 0 else None,
            'cost_margin_pct': round(
                max(0, (1 - predictions['estimated_cost_usd'] / cap['max_cost_usd_observed']) * 100), 1
            ) if cap['max_cost_usd_observed'] > 0 else None,
        }
        return {
            'model_version':   self.metadata['model_version'],
            'topology':        self.metadata['topology'],
            'timestamp':       datetime.now().isoformat(),
            'input_state':     state,
            'sla_config':      sla,
            'positioning':     positioning,
            'predictions':     predictions,
            'optimization':    optimization,
            'status':          status,
            'warnings':        warns,
            'recommendations': recs,
            'headroom':        headroom,
        }

    def recommend_from_parquet(self, parquet_dir, sla=None):
        p = Path(parquet_dir)
        df_task  = pd.read_parquet(p / 'task.parquet')
        df_svc   = pd.read_parquet(p / 'service.parquet')
        df_power = pd.read_parquet(p / 'powerSource.parquet')
        MS_TO_S, MS_TO_H, J_TO_KWH = 1_000, 3_600_000, 1/3_600_000
        wait_s = ((df_task['schedule_time'] - df_task['submission_time']) / MS_TO_S).clip(lower=0)
        exec_s = ((df_task['finish_time']   - df_task['schedule_time'])   / MS_TO_S).clip(lower=0)
        dur_h  = df_task['finish_time'].max() / MS_TO_H
        state = {
            'n_tasks_active':       int(len(df_task)),
            'exec_mean_s':          float(exec_s.mean()),
            'sim_duration_h':       float(dur_h),
            'energy_consumed_kwh':  float(df_power['energy_usage'].iloc[-1] * J_TO_KWH),
            'wait_mean_s':          float(wait_s.mean()),
            'hosts_active':         int(df_svc['hosts_up'].iloc[-1]) if 'hosts_up' in df_svc.columns else None,
            'tasks_completed':      int(df_svc['tasks_terminated'].iloc[-1]) if 'tasks_terminated' in df_svc.columns else None,
            'source': 'parquet', 'parquet_dir': str(p),
        }
        return self.recommend(state, sla)


print('✅ Classe RecommendationModel definida')

In [ ]:
# ── Bloco 11 (Célula 22 original): Instanciar e salvar o modelo ───────────

metadata_json = {
    'model_version':       MODEL_VERSION,
    'topology':            TOPOLOGY,
    'trained_on':          datetime.now().isoformat(),
    'n_scenarios':         len(df),
    'feature_cols':        FEATURE_COLS,
    'target_cols':         TARGET_COLS,
    'primary_predictor':   'n_unique_tasks',
    'positioning_method':  'euclidean_knn_k3_normalized',
    'validation': {
        'method': 'leave-one-out',
        'results': df_val.reset_index().to_dict(orient='records'),
    },
    'pricing': {
        'price_usd_per_h': EC2_PRICE_USD_PER_H,
        'instance_type':   TOPOLOGY,
        'pricing_model':   'On-Demand',
        'citation':        PRICING_CITATION,
        'bibtex':          PRICING_BIBTEX,
        'source':          'AWS Pricing API (public endpoint, no authentication required)',
        'note':            'Preco carregado de pricing/pricing_metadata.json',
    },
    'artifact_format':       'json',
    'compatible_runtimes':   ['python', 'node.js', 'any'],
    'notes': (
        'Modelo treinado com 7 cenarios (n_unique_tasks in [10, 350]). '
        'Extrapolacao alem desse range requer cautela. '
        'Sobre-provisionamento silencioso: cpu_util ~0% em todos os cenarios — '
        'o otimizador dimensiona por demanda de compute (Lei de Little), nao por regressao de hosts.'
    ),
}

model = RecommendationModel(
    scaler_params    = scaler_json,
    predictor_params = predictor_json,
    positioner_params= positioner_json,
    scenario_db      = scenario_db,
    rules            = rules_json,
    metadata         = metadata_json,
    optimizer        = optimizer_json,
)
model.save(MODEL_DIR)

In [ ]:
# ── Bloco 12: Round-trip test (dump → load → recommend) ───────────────────
model2 = RecommendationModel.load(MODEL_DIR)

test_state = {
    'n_tasks_active': 50, 'exec_mean_s': 0.22, 'sim_duration_h': 0.002,
    'energy_consumed_kwh': 0.016, 'wait_mean_s': 0.45, 'hosts_active': 100
}
res1 = model.recommend(test_state)
res2 = model2.recommend(test_state)

for target in TARGET_COLS:
    v1 = res1['predictions'][target]
    v2 = res2['predictions'][target]
    assert abs(v1 - v2) < 1e-10, f'Round-trip mismatch para {target}: {v1} vs {v2}'

print('✅ Round-trip OK (diff < 1e-10 em todas as predições)')
print(f'   Optimizer carregado: {"sim" if model2.optimizer else "nao"}')
print(f'   Exemplo de optimization block:')
opt_ex = res2.get('optimization', {})
print(f'   current_hosts = {opt_ex.get("current_config", {}).get("hosts")}')
print(f'   recommended   = {opt_ex.get("recommended_config", {}).get("hosts")}')
print(f'   feasible      = {opt_ex.get("feasible")}')

In [ ]:
# ── Bloco 13 (Célula 24): Testes de aceite + validação dos 7 cenários ─────
print('=' * 65)
print('CRITÉRIO DE ACEITE — estado de referência (110 tasks)')
print('=' * 65)

ref_state = {
    'n_tasks_active': 110, 'exec_mean_s': 0.22, 'sim_duration_h': 0.0015,
    'energy_consumed_kwh': 0.011, 'wait_mean_s': 0.44, 'hosts_active': 100
}
ref_sla = {'max_energy_kwh': 0.015, 'max_wait_s': 0.55, 'max_cost_usd': 0.008}

opt = model.recommend(ref_state, ref_sla)['optimization']
print(f'  window_s         = {ref_state["sim_duration_h"] * 3600:.2f} s')
print(f'  concurrency      = {opt["demand"]["concurrency_tasks"]:.4f} tasks')
print(f'  cores_required   = {opt["demand"]["cores_required"]:.3f} cores')
print(f'  recommended_hosts= {opt["recommended_config"]["hosts"]}')
print(f'  util_at_rec      = {opt["demand"]["utilization_at_recommended_pct"]:.2f}%')
print(f'  savings_hosts    = {opt["savings"]["hosts"]}')
print(f'  savings_pct      = {opt["savings"]["pct"]:.2f}%')
print(f'  feasible         = {opt["feasible"]}')
print(f'  cost_opt         = ${opt["projected"]["cost_usd"]:.8f}')
print(f'  energy_opt       = {opt["projected"]["energy_kwh"]:.8f} kWh')

assert opt['recommended_config']['hosts'] <= 6,  f"hosts={opt['recommended_config']['hosts']} (esperado ≤ 6)"
assert opt['savings']['pct'] >= 90,              f"savings={opt['savings']['pct']}% (esperado ≥ 90%)"
assert opt['feasible'],                          'feasible=False (esperado True)'
print('\n✅ Asserts: recommended ≤ 6 | savings ≥ 90% | feasible=True  PASSARAM')

print('\n─── Validação dos 7 cenários (feasible nunca False) ─────────────────────')
all_feasible = True
for scen in scenario_db:
    s = {
        'n_tasks_active':      scen['n_tasks'],
        'exec_mean_s':         scen['exec_mean_s'],
        'sim_duration_h':      scen['sim_duration_h'],
        'energy_consumed_kwh': scen['energy_total_kwh'],
        'wait_mean_s':         scen['wait_mean_s'],
        'hosts_active':        scen['n_hosts'],
    }
    r    = model.find_optimal_config(s)
    icon = '✅' if r['feasible'] else '❌'
    if not r['feasible']: all_feasible = False
    print(
        f"  {icon} {scen['scenario']:<18} → "
        f"{r['recommended_config']['hosts']:>4} hosts | "
        f"economia {r['savings']['pct']:>6.1f}% | "
        f"feasible={r['feasible']}"
    )
assert all_feasible, 'Algum cenário retornou feasible=False!'
print('\n✅ Todos os 7 cenários: feasible=True')

In [ ]:
# ── Bloco 14: Sumário final ────────────────────────────────────────────────
print('=' * 65)
print('MODELO SALVO EM model_bundle/')
print('=' * 65)
for f in sorted(MODEL_DIR.glob('*.json')):
    print(f'  {f.name}')

print()
print(f'n_scenarios       : {len(df)}')
print(f'feature_cols      : {FEATURE_COLS}')
print(f'target_cols       : {TARGET_COLS}')
print(f'avg_cores_per_task: {AVG_CORES_PER_TASK}')
print(f'price_usd_per_h   : {EC2_PRICE_USD_PER_H}')
print(f'target_util       : {TARGET_UTIL}')

print()
print('Próximos passos:')
print('  1. O digital-twin-api consome model_bundle/ (já implementado)')
print('  2. O collector TypeScript deve consumir tópicos Redpanda (próxima etapa)')
print('  3. Comparar example_recommendation.json com saída histórica para validação final')